# 18 — Final trio dashboard

This notebook validates and renders one completed canonical trio report bundle. It
does not rebuild a market snapshot, Factor run, SJM run, or canonical report table.
Every plotted value comes from the hash-inventoried `tear_sheet_trio_ext2026` Parquet
asset after row-level validation.

Optional overrides:
- `FINANCE_NOTEBOOK_REPORT_ROOT` — completed `canonical_reports.v1` directory
  (preferred).
- `FINANCE_NOTEBOOK_SOURCE_ROOT` — backward-compatible alias for the same completed
  report directory.
- `FINANCE_NOTEBOOK_OUTPUT_DIR` — directory for the three presentation PNGs;
  defaults to `data/`.
- `FINANCE_NOTEBOOK_REPO_ROOT` — project root when executing a temporary notebook
  copy outside the repository.

Run with the project environment (`uv run jupyter` or `./start_jupyter_lab.sh`) and a
fresh kernel.


In [1]:
import hashlib
import json
import os
import re
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


def _repository_root() -> Path:
    configured = os.environ.get("FINANCE_NOTEBOOK_REPO_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / "pyproject.toml").is_file() or not (root / "notebooks").is_dir():
            raise ValueError(
                "FINANCE_NOTEBOOK_REPO_ROOT must name the project directory containing "
                f"pyproject.toml and notebooks/: {root}"
            )
        return root
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (candidate / "pyproject.toml").exists() and (candidate / "notebooks").exists():
            return candidate.resolve()
    return Path.cwd().resolve()


REPO = _repository_root()
sys.path.insert(0, str(REPO))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from macro_framework.reporting import READER_SCHEMA, validate_report_row


TRIO_STEM = "tear_sheet_trio_ext2026"
PROVISIONAL_REPORT_ROOT = REPO / "data" / "provisional_remediation" / "canonical_reports"
_DATE_COLUMNS = {
    "start",
    "end",
    "actual_end",
    "anchor",
    "first_return_date",
    "requested_start",
    "requested_end",
    "raw_market_model_start",
    "raw_market_model_end",
}


def _resolve_path(value: str) -> Path:
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (REPO / path).resolve()


def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def _report_root() -> Path:
    configured = os.environ.get("FINANCE_NOTEBOOK_REPORT_ROOT") or os.environ.get(
        "FINANCE_NOTEBOOK_SOURCE_ROOT"
    )
    root = _resolve_path(configured) if configured else PROVISIONAL_REPORT_ROOT
    if not root.is_dir():
        raise FileNotFoundError(
            "completed canonical report root is missing: "
            f"{root}. Set FINANCE_NOTEBOOK_REPORT_ROOT to a completed canonical_reports.v1 directory."
        )
    return root


def _output_dir() -> Path:
    configured = os.environ.get("FINANCE_NOTEBOOK_OUTPUT_DIR")
    output = _resolve_path(configured) if configured else REPO / "data"
    output.mkdir(parents=True, exist_ok=True)
    if not output.is_dir():
        raise ValueError(f"presentation output path is not a directory: {output}")
    return output


def _identity_entry(
    manifest: dict[str, object], family: str, identity_key: str
) -> tuple[str, str]:
    inputs = manifest.get("input_manifests")
    if not isinstance(inputs, dict) or not isinstance(inputs.get(family), dict):
        raise ValueError(f"canonical report manifest has no {family!r} lineage entry")
    entry = inputs[family]
    identity = entry.get(identity_key)
    digest = entry.get("manifest_sha256")
    if not isinstance(identity, str) or not identity:
        raise ValueError(f"canonical report {family!r} identity is invalid")
    if not isinstance(digest, str) or re.fullmatch(r"[0-9a-f]{64}", digest) is None:
        raise ValueError(f"canonical report {family!r} manifest SHA-256 is invalid")
    return identity, digest


def load_completed_trio(root: Path) -> tuple[dict[str, object], pd.DataFrame, Path]:
    manifest_path = root / "manifest.json"
    marker_path = root / "COMPLETED"
    if not manifest_path.is_file() or not marker_path.is_file():
        raise ValueError(f"{root}: completed canonical report manifest or marker is missing")
    manifest = json.loads(manifest_path.read_text())
    if not isinstance(manifest, dict):
        raise ValueError(f"{manifest_path}: manifest must be a JSON object")
    if manifest.get("schema") != "canonical_reports.v1" or manifest.get("completed") is not True:
        raise ValueError(f"{manifest_path}: expected completed canonical_reports.v1 manifest")
    marker = f"manifest_sha256={sha256_file(manifest_path)}"
    if marker not in marker_path.read_text().splitlines():
        raise ValueError(f"{root}: COMPLETED marker does not match manifest bytes")

    tables = manifest.get("tables")
    entry = tables.get(TRIO_STEM) if isinstance(tables, dict) else None
    if not isinstance(entry, dict):
        raise ValueError(f"{manifest_path}: missing inventory entry for {TRIO_STEM}")
    relative_file, expected_sha, expected_rows = (
        entry.get("file"),
        entry.get("sha256"),
        entry.get("rows"),
    )
    if not isinstance(relative_file, str) or not isinstance(expected_sha, str):
        raise ValueError(f"{manifest_path}: {TRIO_STEM} inventory is incomplete")
    if isinstance(expected_rows, bool) or not isinstance(expected_rows, int) or expected_rows < 1:
        raise ValueError(f"{manifest_path}: {TRIO_STEM} row count is invalid")
    artifact = (root / relative_file).resolve()
    try:
        artifact.relative_to(root.resolve())
    except ValueError as exc:
        raise ValueError(f"{manifest_path}: {TRIO_STEM} path escapes report root") from exc
    if artifact.suffix != ".parquet" or not artifact.is_file():
        raise ValueError(f"{manifest_path}: {TRIO_STEM} must be an existing Parquet asset")
    if sha256_file(artifact) != expected_sha:
        raise ValueError(f"{artifact}: bytes do not match the canonical report manifest")

    frame = normalize_table(pd.read_parquet(artifact))
    if len(frame) != expected_rows:
        raise ValueError(
            f"{artifact}: expected {expected_rows} canonical trio rows, found {len(frame)}"
        )
    return manifest, frame, artifact


def normalize_table(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    for column in out.columns:
        if column in _DATE_COLUMNS or column.endswith("_date"):
            out[column] = pd.to_datetime(out[column], format="mixed", errors="raise")
    return out


def _scalar_or_none(value):
    if isinstance(value, np.generic):
        value = value.item()
    if value is None or value is pd.NA or value is pd.NaT:
        return None
    return value


def validated_trio_rows(frame: pd.DataFrame) -> list[dict[str, object]]:
    if "schema" not in frame or set(frame["schema"]) != {READER_SCHEMA}:
        raise ValueError(f"canonical trio must contain only {READER_SCHEMA} rows")
    rows = [
        validate_report_row(
            {key: _scalar_or_none(value) for key, value in raw.items()}
        )
        for raw in frame.to_dict(orient="records")
    ]
    if len(rows) != 3 or any(row["row_kind"] != "full" for row in rows):
        raise ValueError("canonical trio must contain exactly three full reader rows")
    signatures = {
        (row["start"], row["end"], row["n_obs"], row["periods_per_year"])
        for row in rows
    }
    if len(signatures) != 1:
        raise ValueError("canonical trio reader rows do not share one performance signature")
    if len({row["cash_benchmark_id"] for row in rows}) != 1 or len(
        {row["currency_basis"] for row in rows}
    ) != 1:
        raise ValueError("canonical trio reader rows do not share cash and currency identities")
    if any("#" not in str(row["source"]) for row in rows):
        raise ValueError("canonical trio reader rows require hash-bound source provenance")
    return rows


def canonical_trio_view(rows: list[dict[str, object]]) -> pd.DataFrame:
    display_rows: list[dict[str, object]] = []
    for cleaned in rows:
        ann_vol = float(cleaned["ann_vol"])
        r2 = float(cleaned["raw_market_model_r2"])
        if not np.isfinite(ann_vol) or ann_vol <= 0.0:
            raise ValueError(f"{cleaned['portfolio_id']}: annualized volatility must be finite and positive")
        if not np.isfinite(r2) or not 0.0 <= r2 < 1.0:
            raise ValueError(f"{cleaned['portfolio_id']}: raw market-model R² must be in [0, 1)")
        residual_vol = ann_vol * float((1.0 - r2) ** 0.5)
        if residual_vol <= 0.0:
            raise ValueError(f"{cleaned['portfolio_id']}: residual volatility must be positive")
        display_rows.append(
            {
                "portfolio_id": str(cleaned["portfolio_id"]),
                "start": pd.Timestamp(cleaned["start"]),
                "end": pd.Timestamp(cleaned["end"]),
                "n_obs": int(cleaned["n_obs"]),
                "total_return": float(cleaned["total_return"]),
                "cagr": float(cleaned["cagr"]),
                "ann_vol": ann_vol,
                "cagr_over_vol": float(cleaned["cagr"]) / ann_vol,
                "sharpe": float(cleaned["sharpe"]),
                "maxdd": float(cleaned["maxdd"]),
                "calmar": float(cleaned["calmar"]),
                "alpha_ann": float(cleaned["raw_market_model_intercept_ann_arithmetic"]),
                "residual_vol_ann": residual_vol,
                "appraisal": float(cleaned["raw_market_model_intercept_ann_arithmetic"]) / residual_vol,
                "r2": r2,
                "cash_benchmark_id": str(cleaned["cash_benchmark_id"]),
                "currency_basis": str(cleaned["currency_basis"]),
                "source": str(cleaned["source"]),
                "ssr": float(cleaned["ssr_ssr"]),
            }
        )
    return pd.DataFrame(display_rows)


REPORT_ROOT = _report_root()
PRESENTATION_OUTPUT_DIR = _output_dir()


In [2]:
canonical_manifest, trio_loaded, trio_path = load_completed_trio(REPORT_ROOT)

factor_run_id, factor_manifest_sha = _identity_entry(
    canonical_manifest, "factor_run", "run_id"
)
overlay_id, sjm_manifest_sha = _identity_entry(canonical_manifest, "sjm_run", "run_id")
snapshot_id, snapshot_manifest_sha = _identity_entry(
    canonical_manifest, "market_snapshot", "snapshot_id"
)
factor_id = "factor_pit_ext2026"

reader_rows = validated_trio_rows(trio_loaded)
rows_by_id = {str(row["portfolio_id"]): row for row in reader_rows}
portfolio_ids = set(rows_by_id)
if len(portfolio_ids) != 3 or {factor_id, overlay_id} - portfolio_ids:
    raise ValueError(
        "canonical trio portfolio identities do not match the required Factor PIT and SJM overlay lines"
    )
if not str(rows_by_id[factor_id]["source"]).startswith("scripts/extend_stream_2026.py:"):
    raise ValueError("canonical Factor PIT row is not sourced by the Factor producer")
if not str(rows_by_id[overlay_id]["source"]).startswith(f"sjm_run:{overlay_id}/"):
    raise ValueError("canonical SJM row does not match the pinned SJM run identity")
static_ids = portfolio_ids - {factor_id, overlay_id}
if len(static_ids) != 1:
    raise ValueError("canonical trio must contain exactly one static comparison row")
static_id = static_ids.pop()

DISPLAY_NAMES = {
    static_id: "Static B&H 19-26",
    factor_id: "Factor ext26",
    overlay_id: "SJM v3 overlay",
}
PLOT_ORDER = [static_id, factor_id, overlay_id]
COLORS = {
    static_id: "#2f6db3",
    factor_id: "#e8710a",
    overlay_id: "#8656c9",
}

trio_rows = canonical_trio_view(reader_rows).set_index("portfolio_id").loc[PLOT_ORDER].reset_index()
trio_rows["display_name"] = trio_rows["portfolio_id"].map(DISPLAY_NAMES)
trio_by_id = trio_rows.set_index("portfolio_id")

for portfolio_id in PLOT_ORDER:
    for field in (
        "cagr",
        "ann_vol",
        "sharpe",
        "maxdd",
        "calmar",
        "alpha_ann",
        "residual_vol_ann",
        "appraisal",
        "ssr",
    ):
        value = float(trio_by_id.loc[portfolio_id, field])
        if not np.isfinite(value):
            raise ValueError(f"{portfolio_id}: {field} is not finite")

display(
    trio_rows[
        [
            "display_name",
            "start",
            "end",
            "n_obs",
            "total_return",
            "cagr",
            "ann_vol",
            "sharpe",
            "maxdd",
            "calmar",
            "alpha_ann",
            "residual_vol_ann",
            "appraisal",
            "ssr",
        ]
    ]
)
print("canonical report root:", REPORT_ROOT)
print("canonical trio table:", trio_path, sha256_file(trio_path))
print("market snapshot:", snapshot_id, snapshot_manifest_sha)
print("factor bundle:", factor_run_id, factor_manifest_sha)
print("sjm run:", overlay_id, sjm_manifest_sha)
print("presentation output directory:", PRESENTATION_OUTPUT_DIR)


,display_name,start,end,n_obs,total_return,cagr,ann_vol,sharpe,maxdd,calmar,alpha_ann,residual_vol_ann,appraisal,ssr
0,Static B&H 19-26,2019-01-03,2026-06-30,1845,2.401641,0.177550,0.136449,1.099819,-0.178468,0.994853,0.071336,0.077840,0.916436,0.143575
1,Factor ext26,2019-01-03,2026-06-30,1845,1.468075,0.128182,0.092823,1.090813,-0.114053,1.123881,0.084532,0.080621,1.048502,0.133262
2,SJM v3 overlay,2019-01-03,2026-06-30,1845,1.213649,0.111914,0.078984,1.078992,-0.088272,1.267837,0.079823,0.071329,1.119088,0.127859


canonical report root: /home/mc/projects/Global_Macro_AI_Factors/data/provisional_remediation/canonical_reports_devstartfix_full_20260730T154855Z
canonical trio table: /home/mc/projects/Global_Macro_AI_Factors/data/provisional_remediation/canonical_reports_devstartfix_full_20260730T154855Z/tables/tear_sheet_trio_ext2026.parquet 1380f9aa2ed77e7b86154d95407e0b9fbd32d92a52f6b02af450310e780497cf
market snapshot: provisional_market_total_return_fx_2026-06-30_v1 d56e1d0a17dd3ad05747dd58b829ba3fbf6b56ca46d2f744fc4672ebb8b22a09
factor bundle: factor_ext2026_2019-01-01_2026-06-30_v1 ef38f37f6798773f366b8631a1205cd945e03044d597fd281a9acf94adf33278
sjm run: sjm_crowding_v3_total_return_bil_provisional_devstartfix_20260730T154855Z 070c5f371cc3ea5083c59300f03721fc12f7a5025494668179b9b429748ea0d4
presentation output directory: /home/mc/projects/Global_Macro_AI_Factors/data


## 1. Risk–return maps — the canonical trio on one validated window

Both panels use the same canonical trio rows. The left panel plots CAGR against total
volatility. The right panel plots the published raw market-model intercept against
residual volatility derived from the same row's R² field.


In [3]:
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
INK, MUTED = "#333333", "#8a8a8a"


def ray_panel(ax, slopes, slope_fmt, xmax, ymax, frac):
    ax.set_xlim(0, xmax)
    ax.set_ylim(0, ymax)
    ax.grid(alpha=0.25, zorder=0)
    for slope in slopes:
        ax.plot([0, xmax], [0, slope * xmax], color="#cccccc", lw=1, zorder=1)
        x_end = min(xmax, ymax / slope)
        p0 = ax.transData.transform((0.0, 0.0))
        p1 = ax.transData.transform((x_end, slope * x_end))
        angle = np.degrees(np.arctan2(p1[1] - p0[1], p1[0] - p0[0]))
        ax.annotate(
            slope_fmt.format(slope),
            (frac * x_end, frac * slope * x_end),
            color=MUTED,
            fontsize=8,
            ha="center",
            va="bottom",
            rotation=angle,
            rotation_mode="anchor",
            xytext=(0, 2),
            textcoords="offset points",
        )


fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(11.5, 5.2))

ray_panel(ax_a, (1.0, 1.5, 2.0), "CAGR/σ = {:.1f}", 20.5, 31.0, frac=0.55)
offsets_a = {
    static_id: (-10, 14),
    factor_id: (10, 10),
    overlay_id: (-13, -4),
}
for portfolio_id in PLOT_ORDER:
    row = trio_by_id.loc[portfolio_id]
    x = float(row["ann_vol"]) * 100.0
    y = float(row["cagr"]) * 100.0
    ax_a.scatter(x, y, s=150, color=COLORS[portfolio_id], edgecolor="white", linewidth=1.6, zorder=3, label=DISPLAY_NAMES[portfolio_id])
    dx, dy = offsets_a[portfolio_id]
    ax_a.annotate(
        f"{DISPLAY_NAMES[portfolio_id]}\nCAGR/σ {row['cagr_over_vol']:.2f}",
        (x, y),
        xytext=(dx, dy),
        textcoords="offset points",
        fontsize=8.5,
        color=INK,
        ha="center" if dx == 0 else ("left" if dx > 0 else "right"),
    )
ax_a.set_xlabel("annualized volatility σ (%)")
ax_a.set_ylabel("CAGR (%)")
ax_a.set_title("Total risk: CAGR vs volatility", fontsize=11)

ray_panel(ax_b, (1.0, 1.25, 1.5, 1.75), "intercept/σε = {:.2f}", 11.5, 17.5, frac=0.50)
offsets_b = {
    static_id: (0, -24),
    factor_id: (10, -14),
    overlay_id: (-10, 10),
}
for portfolio_id in PLOT_ORDER:
    row = trio_by_id.loc[portfolio_id]
    x = float(row["residual_vol_ann"]) * 100.0
    y = float(row["alpha_ann"]) * 100.0
    ax_b.scatter(x, y, s=150, color=COLORS[portfolio_id], edgecolor="white", linewidth=1.6, zorder=3)
    dx, dy = offsets_b[portfolio_id]
    ax_b.annotate(
        f"{DISPLAY_NAMES[portfolio_id]}\nAppraisal {row['appraisal']:.2f}",
        (x, y),
        xytext=(dx, dy),
        textcoords="offset points",
        fontsize=8.5,
        color=INK,
        ha="center" if dx == 0 else ("left" if dx > 0 else "right"),
    )
ax_b.set_xlabel("regression residual volatility σε (%)")
ax_b.set_ylabel("regression intercept (%, ann.)")
ax_b.set_title("Regression-adjusted: intercept vs residual risk", fontsize=11)

fig.legend(loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("Return per unit of risk — steeper ray = better", y=1.08, fontsize=12)
fig.tight_layout()
output_path = PRESENTATION_OUTPUT_DIR / "nb18_risk_return_maps.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

print("figure:", output_path)

figure: /home/mc/projects/Global_Macro_AI_Factors/data/nb18_risk_return_maps.png


### Reading the risk–return maps

The colored dots are the three canonical strategy rows. On the left, each gray ray
holds CAGR divided by total volatility constant; on the right, it holds the published
raw market-model intercept divided by residual volatility constant. A steeper ray
therefore means a larger plotted return-per-risk ratio in that panel. The left ratio
is descriptive CAGR/volatility, not BIL-excess Sharpe: it neither subtracts BIL nor
uses mean excess return. The right ratio is the appraisal ratio based on intercept
and residual volatility. These rays are reference lines only. They are not portfolios,
an efficient frontier, or capital allocation lines.


## 2. Ratio ladder — the same three lines, three denominators

Every dot comes from the validated canonical trio rows. The ladder expresses those
rows as return per unit of total volatility, drawdown, and residual regression risk.


In [4]:
ratios = [
    ("Sharpe (return / total vol)", "sharpe"),
    ("Calmar (CAGR / |max drawdown|)", "calmar"),
    ("Appraisal (intercept / residual vol)", "appraisal"),
]

fig, ax = plt.subplots(figsize=(9.5, 3.6))
for i, (row_label, key) in enumerate(ratios):
    y = len(ratios) - 1 - i
    xs = [float(trio_by_id.loc[portfolio_id, key]) for portfolio_id in PLOT_ORDER]
    ax.plot([min(xs), max(xs)], [y, y], color="#d5d5d5", lw=2, zorder=1)
    for portfolio_id in PLOT_ORDER:
        row = trio_by_id.loc[portfolio_id]
        ax.scatter(
            float(row[key]),
            y,
            s=140,
            color=COLORS[portfolio_id],
            edgecolor="white",
            linewidth=1.5,
            zorder=3,
            label=DISPLAY_NAMES[portfolio_id] if i == 0 else None,
        )
        ax.annotate(
            f"{float(row[key]):.2f}",
            (float(row[key]), y),
            xytext=(0, 11),
            textcoords="offset points",
            ha="center",
            fontsize=8.5,
            color=INK,
        )

all_values = [float(trio_by_id.loc[portfolio_id, key]) for portfolio_id in PLOT_ORDER for _, key in ratios]
pad = 0.18 * (max(all_values) - min(all_values))
ax.set_yticks(range(len(ratios)), [label for label, _ in reversed(ratios)])
ax.set_xlabel("return per unit of risk (unitless ratio)")
ax.set_xlim(min(all_values) - pad, max(all_values) + pad)
ax.set_ylim(-0.6, 2.65)
ax.grid(axis="x", alpha=0.25, zorder=0)
ax.legend(loc="upper left", frameon=False, fontsize=9)
ax.set_title("Return per unit of risk — three denominators", fontsize=11, pad=14)
fig.tight_layout()
output_path = PRESENTATION_OUTPUT_DIR / "nb18_ratio_ladder.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

print("figure:", output_path)

figure: /home/mc/projects/Global_Macro_AI_Factors/data/nb18_ratio_ladder.png


In [5]:
# Metric profile — six canonical axes
profile = [
    ("CAGR", "cagr", "{:.1%}", False),
    ("Intercept (ann.)", "alpha_ann", "{:.1%}", False),
    ("Max DD", "maxdd", "{:.1%}", False),
    ("Residual vol σε", "residual_vol_ann", "{:.1%}", True),
    ("Sharpe", "sharpe", "{:.2f}", False),
    ("Appraisal", "appraisal", "{:.2f}", False),
]

fig, ax = plt.subplots(figsize=(11.5, 5.4))
positions = {}
for j, (name, key, fmt, invert) in enumerate(profile):
    raw = np.array([float(trio_by_id.loc[portfolio_id, key]) for portfolio_id in PLOT_ORDER])
    scaled = np.full(len(raw), 0.5) if np.ptp(raw) == 0 else (raw - raw.min()) / np.ptp(raw)
    if invert:
        scaled = 1 - scaled
    rel = np.ptp(raw) / max(1e-12, np.mean(np.abs(raw)))
    half = max(0.05, 0.40 * min(1.0, rel / 0.40))
    positions[j] = 0.5 + (scaled - 0.5) * 2 * half
    ax.axvline(j, color="#dddddd", lw=1, zorder=0)

dodge = {0: -0.045, 1: 0.0, 2: 0.045}
for i, portfolio_id in enumerate(PLOT_ORDER):
    xs = [j + dodge[i] for j in range(len(profile))]
    ys = [positions[j][i] for j in range(len(profile))]
    ax.plot(xs, ys, color=COLORS[portfolio_id], lw=2, zorder=2, alpha=0.85, label=DISPLAY_NAMES[portfolio_id])
    ax.scatter(xs, ys, s=52, color=COLORS[portfolio_id], edgecolor="white", linewidth=1.2, zorder=3)
    lx, ly = [(-30, 6), (0, -13), (30, 6)][i]
    for j, (_, key, fmt, _) in enumerate(profile):
        ax.annotate(
            fmt.format(float(trio_by_id.loc[portfolio_id, key])),
            (xs[j], ys[j]),
            xytext=(lx, ly),
            textcoords="offset points",
            ha="center",
            va="bottom" if ly > 0 else "top",
            fontsize=7.5,
            color=INK,
        )

ax.set_xticks(range(len(profile)), [item[0] for item in profile], fontsize=9.5)
ax.set_yticks([])
ax.set_ylim(0, 1.02)
ax.set_xlim(-0.45, len(profile) - 0.55)
for side in ("left", "bottom"):
    ax.spines[side].set_visible(False)
ax.legend(loc="lower center", ncol=3, frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.16))
ax.set_title(
    "Metric profile — up = preferable; vertical separation tracks relative spread",
    fontsize=11,
)
fig.tight_layout()
output_path = PRESENTATION_OUTPUT_DIR / "nb18_metric_profile.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

print("figure:", output_path)
print("canonical source strings checked before plotting:")
for portfolio_id in PLOT_ORDER:
    row = trio_by_id.loc[portfolio_id]
    print(f"  {DISPLAY_NAMES[portfolio_id]}: {row['source']}")

figure: /home/mc/projects/Global_Macro_AI_Factors/data/nb18_metric_profile.png
canonical source strings checked before plotting:
  Static B&H 19-26: scripts/build_tear_sheet.py:static_bh_25pct|market_snapshot:provisional_market_total_return_fx_2026-06-30_v1/basket_adjusted_close_local.parquet#90993b1a5f1f05c1126d4ba52a5849baa6cd610047f2e3f90119d054dffe76cf|market_snapshot:provisional_market_total_return_fx_2026-06-30_v1/cash_market_total_return.parquet#BIL@87a8f564aefb4534b692bf5cba693264639bfb88f44e4a32a18655bb7689f0b3
  Factor ext26: scripts/extend_stream_2026.py:factor_equity_ext2026.parquet|market_snapshot:provisional_market_total_return_fx_2026-06-30_v1/cash_market_total_return.parquet#BIL@87a8f564aefb4534b692bf5cba693264639bfb88f44e4a32a18655bb7689f0b3|market_snapshot:provisional_market_total_return_fx_2026-06-30_v1/cash_market_total_return.parquet#SPY@87a8f564aefb4534b692bf5cba693264639bfb88f44e4a32a18655bb7689f0b3
  SJM v3 overlay: sjm_run:sjm_crowding_v3_total_return_bil_provi

## Reading guide

- All three figures consume the one hash-inventoried canonical trio Parquet table.
  Its completed manifest, row count, SHA-256, row schemas, common window, cash
  benchmark, currency basis, and source hashes are checked before plotting.
- The canonical bundle pins the Factor, SJM v3, and market-snapshot manifest
  identities printed above. The dashboard does not re-open or rebuild those
  producer artifacts.
- Residual volatility and the appraisal ratio are derived only from each
  validated row's published intercept and R² fields.
- Only presentation figures are written, to `FINANCE_NOTEBOOK_OUTPUT_DIR` (or
  `data/` by default): `nb18_risk_return_maps.png`, `nb18_ratio_ladder.png`,
  and `nb18_metric_profile.png`.